# Where — and when — does a diffusion robot policy look?
**Faithful visual attribution for RDT-1B on ManiSkill (pilot study)**

Runs the study planned in
[`docs/GAMEPLAN.md`](https://github.com/rdgbrandon/rdt-igtesting/blob/cvpr-xai/docs/GAMEPLAN.md) and produces the
paper's tables and figures. **Run [`rdt_repro_check.ipynb`](https://colab.research.google.com/github/rdgbrandon/rdt-igtesting/blob/cvpr-xai/notebooks/rdt_repro_check.ipynb)
first** and set `ENV` in cell 3 to the environment where RDT reproduces its paper success rate.

| Stage | What it measures |
|---|---|
| **verify** | cached sampler == RDT's own; rendering works; measured speed → projected runtime |
| **A** Rollouts | RDT success rate with the official protocol (should match 77% / 74% / 100%) |
| **B** Benchmark | 11 attribution methods at matched compute: deletion / insertion faithfulness, seed stability |
| **C** Denoising time | *when* in the 5 denoising steps vision is used (exact per-step Expected-IG split + knockout) |
| **D** Closed loop | success when the top-10% attributed patches are hidden **while the robot acts** |
| **E** Sanity | maps must change when RDT's weights are randomized (Adebayo et al. 2018) |
| **F** Language & modality | words × joints (word removal + IG); vision / language / proprioception Shapley shares |

Every stage runs as its own process in the chosen environment, saves as it goes and resumes after a disconnect
(just *Run all* again). Results go to Google Drive.

In [ ]:
#@title 1. Setup: GPU, Google Drive, install. Re-running this cell always pulls the latest fixes.
import os, sys, subprocess, importlib
OUT_DRIVE = '/content/drive/MyDrive/rdt-xai/pilot2'  #@param {type:"string"}
from google.colab import drive
drive.mount('/content/drive')
if not os.path.exists('/content/rdt-igtesting'):
    subprocess.run('git clone -q -b cvpr-xai https://github.com/rdgbrandon/rdt-igtesting /content/rdt-igtesting',
                   shell=True, check=True)
subprocess.run('cd /content/rdt-igtesting && git fetch -q origin cvpr-xai && git checkout -q cvpr-xai && git pull -q origin cvpr-xai',
               shell=True, check=True)
sys.path.insert(0, '/content/rdt-igtesting')
import rdt_xai.colab as colab
importlib.reload(colab)
OUT = colab.setup(OUT_DRIVE)

In [ ]:
#@title 2. Download models (~6 GB; skipped if present)
import shutil
from huggingface_hub import hf_hub_download
D = '/content/rdt-data'
for sub in ('rdt', 'siglip', 'lang'):
    os.makedirs(f'{D}/{sub}', exist_ok=True)
for f in ['config.json', 'preprocessor_config.json', 'special_tokens_map.json', 'spiece.model',
          'tokenizer.json', 'tokenizer_config.json', 'model.safetensors']:
    hf_hub_download('google/siglip-so400m-patch14-384', f, local_dir=f'{D}/siglip')
hf_hub_download('robotics-diffusion-transformer/maniskill-model', 'rdt/mp_rank_00_model_states.pt', local_dir=D)
for t in ['PickCube', 'StackCube', 'PushCube', 'PegInsertionSide', 'PlugCharger']:
    p = hf_hub_download('robotics-diffusion-transformer/maniskill-model', f'lang_embeds/text_embed_{t}-v1.pt', local_dir=D)
    if not os.path.exists(f'{D}/lang/text_embed_{t}-v1.pt'):
        shutil.copy(p, f'{D}/lang/text_embed_{t}-v1.pt')
print('models ready')

In [ ]:
#@title 3. Environment + experiment size
import json
ENV = 'rdt_era'   #@param ["rdt_era", "current"]
#@markdown `rdt_era` = ManiSkill 3.0.0b15 + SAPIEN 3.0.0b1 (Nov 2024, what RDT was evaluated on), Python 3.11.
#@markdown `current` = ManiSkill 3.0.1 + SAPIEN 3.0.3, Colab's Python. Use whichever reproduces RDT in the check.
TASKS = ['PickCube-v1', 'StackCube-v1', 'PushCube-v1']  #@param
N_EPISODES = 16           #@param {type:"integer"}
B_MOMENTS = 60            #@param {type:"integer"}
B_STABILITY_MOMENTS = 24  #@param {type:"integer"}
N_EVALS = 16              #@param {type:"integer"}
FAST = False              #@param {type:"boolean"}
C_MOMENTS = 30            #@param {type:"integer"}
D_TASKS = ['PickCube-v1', 'StackCube-v1']  #@param
D_EPISODES = 10           #@param {type:"integer"}
D_FRAC = 0.10             #@param {type:"number"}
E_MOMENTS = 8             #@param {type:"integer"}
F_MOMENTS = 30            #@param {type:"integer"}

PY = colab.rdt_era_env() if ENV == 'rdt_era' else sys.executable
CFG = f'{OUT}/config.json'
json.dump(dict(out=OUT, tasks=TASKS, n_episodes=N_EPISODES, b_moments=B_MOMENTS, b_stability_moments=B_STABILITY_MOMENTS,
               n_evals=N_EVALS, fast=FAST, c_moments=C_MOMENTS, d_tasks=D_TASKS, d_episodes=D_EPISODES, d_frac=D_FRAC,
               e_moments=E_MOMENTS, f_moments=F_MOMENTS, env=ENV), open(CFG, 'w'), indent=1)
RUN_ENV = dict(os.environ, RDT_REPO='/content/RoboticsDiffusionTransformer', RDT_DATA='/content/rdt-data')


def run_stage(stage, *extra):
    # run one stage in the chosen environment, streaming its progress; raises if it fails
    p = subprocess.Popen([PY, 'scripts/run_stage.py', '--config', CFG, '--stage', stage, *extra],
                         cwd='/content/rdt-igtesting', env=RUN_ENV, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-40:]
        if 'warn' not in line.lower():
            print(line.rstrip(), flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'stage {stage} failed (exit {p.returncode}); last output:\n' + ''.join(tail))
    print('synced', colab.sync(OUT, OUT_DRIVE), 'files to Drive')


print('environment:', ENV, PY)

In [ ]:
#@title 4. verify: exactness, rendering, measured speed -> projected runtime
run_stage('verify')
from IPython.display import Image as IPImage, display
display(IPImage(f'{OUT}/verify_render.png', width=256))

In [ ]:
#@title A. Rollouts (official RDT ManiSkill protocol)
run_stage('A')

In [ ]:
#@title B. Attribution benchmark at matched compute
run_stage('B')

In [ ]:
#@title C. When during denoising does vision matter?
run_stage('C')

In [ ]:
#@title D. Closed-loop causal test: hide 10% of patches online, measure success
run_stage('D')

In [ ]:
#@title E. Model-randomization sanity check
run_stage('E')

In [ ]:
#@title F. Language & modality: word x joint attribution, vision / language / proprioception Shapley
run_stage('F')

In [ ]:
#@title Results: tables + figures (also saved to Drive: report/RESULTS.md, report/*.png)
from rdt_xai.report import build_report
from IPython.display import Markdown
txt = build_report(OUT)
display(Markdown(txt))
for f in ['fig_faithfulness_curves', 'fig_qualitative', 'fig_denoising_steps', 'fig_denoising_step_maps',
          'fig_closed_loop', 'fig_sanity', 'fig_word_joint', 'fig_modality']:
    p = f'{OUT}/report/{f}.png'
    if os.path.exists(p):
        print(f); display(IPImage(p, width=900))
print('synced', colab.sync(OUT, OUT_DRIVE), 'files ->', OUT_DRIVE)

In [ ]:
#@title (Optional) Video: one successful episode with Expected-IG overlay at every decision
EPISODE = ''  #@param {type:"string"}
run_stage('video', '--episode', EPISODE) if EPISODE else run_stage('video')
import glob
from IPython.display import Video
for v in sorted(glob.glob(f'{OUT}/report/video_*'))[-1:]:
    print(v); display(Video(v, embed=True, width=384)) if v.endswith('.mp4') else display(IPImage(v))

## Reading the results

* **A** — if success rates are far below the RDT paper (77% / 74% / 100%), stop: nothing downstream is trustworthy.
* **B** — higher *deletion* and *insertion* = more faithful; `seed_spearman` near 1 = the map does not depend on the
  sampler's random noise; paired Wilcoxon tests compare Expected IG with each method.
* **C** — attribution share and knockout effect per denoising step (step 1 = noisiest).
* **D** — a faithful method's *top* patches should hurt success much more than *random*, its *bottom* patches least.
* **E** — Spearman to the trained model's map after randomizing RDT blocks must drop, or the method is not
  explaining the model.
* **F** — how much each decision depends on camera vs. instruction vs. joint state; which words drive which joint.

Send `report/RESULTS.md` (in your Drive folder) back for the next iteration.